In [ ]:
# --- LOAD DATASET ---
import pandas as pd

path = "data/food_recipes.csv"
df = pd.read_csv(path, nrows=100000)
print(df.shape)
print(df.columns.tolist())
df.head(3)

(8009, 16)
['recipe_title', 'url', 'record_health', 'vote_count', 'rating', 'description', 'cuisine', 'course', 'diet', 'prep_time', 'cook_time', 'ingredients', 'instructions', 'author', 'tags', 'category']


,recipe_title,url,record_health,vote_count,rating,description,cuisine,course,diet,prep_time,cook_time,ingredients,instructions,author,tags,category
0,Roasted Peppers And Mushroom Tortilla Pizza Re...,https://www.archanaskitchen.com/roasted-pepper...,good,434,4.958525,is a quicker version pizza to satisfy your cr...,Mexican,Dinner,Vegetarian,15 M,15 M,Tortillas|Extra Virgin Olive Oil|Garlic|Mozzar...,To begin making the Roasted Peppers And Mushro...,Divya Shivaraman,Party Food Recipes|Tea Party Recipes|Mushroom ...,Pizza Recipes
1,Thakkali Gotsu Recipe | Thakkali Curry | Spicy...,https://www.archanaskitchen.com/tomato-gotsu-r...,good,3423,4.932223,also known as the is a quick and easy to ma...,South Indian Recipes,Lunch,Vegetarian,10 M,20 M,Sesame (Gingelly) Oil|Mustard seeds (Rai/ Kadu...,To begin making Tomato Gotsu Recipe/ Thakkali ...,Archana Doshi,Vegetarian Recipes|Tomato Recipes|South Indian...,Indian Curry Recipes
2,Spicy Grilled Pineapple Salsa Recipe,https://www.archanaskitchen.com/spicy-grilled-...,good,2091,4.945959,Spicy Grilled Pineapple Salsa is a simple reci...,Mexican,Side Dish,Vegetarian,10 M,0 M,Extra Virgin Olive Oil|Pineapple|White onion|R...,To begin making the Spicy Grilled Pineapple Sa...,Archana's Kitchen,Party Starter & Appetizer Recipes|Pineapple Re...,Mexican Recipes


In [2]:
# --- INSPECT MISSING AND DUPLICATES ---
print(df.isna().sum())
print("Duplicate titles:", df.duplicated(subset=["recipe_title"]).sum())
print(df.loc[0, "ingredients"])
print(df.loc[0, "instructions"])
print(df["cuisine"].value_counts().head(10))
print(df["diet"].value_counts().head(10))

recipe_title       0
url                0
record_health      0
vote_count         0
rating             0
description       15
cuisine           66
course           155
diet             151
prep_time         30
cook_time         30
ingredients       12
instructions       0
author             0
tags              79
category           0
dtype: int64
Duplicate titles: 0
Tortillas|Extra Virgin Olive Oil|Garlic|Mozzarella cheese|Red Yellow or Green Bell Pepper (Capsicum)|Onions|Kalmatta olives|Button mushrooms
To begin making the Roasted Peppers And Mushroom Tortilla Pizza recipe, turn your oven setting to broiler and keep it ready.|Grease your cast iron skillet with some olive oil and heat it over high heat.|When it starts to shimmer bring down the heat to medium. Add olive oil and minced garlic to the marinara sauce and stir to combine, keep aside.|Place one tortilla on the cast iron skillet. Spread a thin layer of marinara sauce over the tortilla.|Spread the cheese as needed. When the che

In [3]:
# --- CLEAN AND PARSE ---
import re

df_clean = df.dropna(subset=["ingredients"]).drop_duplicates(subset=["recipe_title"]).copy()

def split_pipe(text):
    return [p.strip() for p in str(text).split("|") if p.strip()]

def normalize_ingredient(ing):
    ing = ing.lower()
    ing = re.sub(r"\(.*?\)", "", ing)      # remove bracketed text
    ing = re.sub(r"[^a-z\s]", " ", ing)    # remove symbols and digits
    ing = re.sub(r"\s+", " ", ing).strip()
    return ing

df_clean["ingredients_raw"] = df_clean["ingredients"].apply(split_pipe)
df_clean["ingredients_clean"] = df_clean["ingredients_raw"].apply(
    lambda lst: [x for x in (normalize_ingredient(i) for i in lst) if x]
)
df_clean["steps"] = df_clean["instructions"].apply(split_pipe)

# keep only English recipes
has_hindi_title = df_clean["recipe_title"].str.contains(r"[\u0900-\u097F]", regex=True)
kept_ratio = df_clean["ingredients_clean"].str.len() / df_clean["ingredients_raw"].str.len()
df_clean = df_clean[
    (~has_hindi_title) & (kept_ratio >= 0.8) & (df_clean["ingredients_clean"].str.len() > 0)
].reset_index(drop=True)

print(df_clean.shape)
df_clean[["recipe_title", "ingredients_raw", "ingredients_clean"]].head(5)

(6731, 19)


,recipe_title,ingredients_raw,ingredients_clean
0,Roasted Peppers And Mushroom Tortilla Pizza Re...,"[Tortillas, Extra Virgin Olive Oil, Garlic, Mo...","[tortillas, extra virgin olive oil, garlic, mo..."
1,Thakkali Gotsu Recipe | Thakkali Curry | Spicy...,"[Sesame (Gingelly) Oil, Mustard seeds (Rai/ Ka...","[sesame oil, mustard seeds, curry leaves, garl..."
2,Spicy Grilled Pineapple Salsa Recipe,"[Extra Virgin Olive Oil, Pineapple, White onio...","[extra virgin olive oil, pineapple, white onio..."
3,Karwar Style Dali Thoy Recipe - Toor dal Curry,"[Arhar dal (Split Toor Dal), Turmeric powder (...","[arhar dal, turmeric powder, salt, dry red chi..."
4,Rajma Kofta In Milk And Poppy Seed Gravy Recipe,"[Rajma (Large Kidney Beans), Cashew nuts, Sult...","[rajma, cashew nuts, sultana raisins, asafoeti..."


In [4]:
# --- SAVE STRUCTURED RECORDS ---
import json

def none_if_nan(x):
    return None if pd.isna(x) else x

records = []
for i, row in df_clean.iterrows():
    records.append({
        "id": int(i),
        "title": row["recipe_title"],
        "ingredients_raw": row["ingredients_raw"],
        "ingredients_clean": row["ingredients_clean"],
        "steps": row["steps"],
        "cuisine": none_if_nan(row["cuisine"]),
        "diet": none_if_nan(row["diet"]),
        "course": none_if_nan(row["course"]),
        "prep_time": none_if_nan(row["prep_time"]),
        "cook_time": none_if_nan(row["cook_time"]),
        "url": row["url"],
    })

with open("recipes_clean.json", "w") as f:
    json.dump(records, f)

print(len(records))
print(records[0])

6731
{'id': 0, 'title': 'Roasted Peppers And Mushroom Tortilla Pizza Recipe', 'ingredients_raw': ['Tortillas', 'Extra Virgin Olive Oil', 'Garlic', 'Mozzarella cheese', 'Red Yellow or Green Bell Pepper (Capsicum)', 'Onions', 'Kalmatta olives', 'Button mushrooms'], 'ingredients_clean': ['tortillas', 'extra virgin olive oil', 'garlic', 'mozzarella cheese', 'red yellow or green bell pepper', 'onions', 'kalmatta olives', 'button mushrooms'], 'steps': ['To begin making the Roasted Peppers And Mushroom Tortilla Pizza recipe, turn your oven setting to broiler and keep it ready.', 'Grease your cast iron skillet with some olive oil and heat it over high heat.', 'When it starts to shimmer bring down the heat to medium. Add olive oil and minced garlic to the marinara sauce and stir to combine, keep aside.', 'Place one tortilla on the cast iron skillet. Spread a thin layer of marinara sauce over the tortilla.', 'Spread the cheese as needed. When the cheese starts to melt, add in the toppings which 

In [5]:
# --- BUILD EMBEDDING TEXT ---
for r in records:
    r["embed_text"] = f"{r['title']}. Ingredients: {', '.join(r['ingredients_clean'])}"

print(records[0]["embed_text"])
print(records[1]["embed_text"])

Roasted Peppers And Mushroom Tortilla Pizza Recipe. Ingredients: tortillas, extra virgin olive oil, garlic, mozzarella cheese, red yellow or green bell pepper, onions, kalmatta olives, button mushrooms
Thakkali Gotsu Recipe | Thakkali Curry | Spicy & Tangy Tomato Gravy. Ingredients: sesame oil, mustard seeds, curry leaves, garlic, pearl onions, tomatoes, tamarind, turmeric powder, salt, jaggery


In [6]:
# --- CREATE EMBEDDINGS AND FAISS INDEX ---
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2", device="mps")  # use "cpu" if mps errors

texts = [r["embed_text"] for r in records]
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,   # makes inner product = cosine similarity
)
embeddings = np.array(embeddings, dtype="float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

faiss.write_index(index, "recipes.faiss")
np.save("recipe_embeddings.npy", embeddings)

print(embeddings.shape, index.ntotal)

/opt/anaconda3/envs/recipe-rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1019 17:46:10.244000 3074 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Batches: 100%|██████████| 106/106 [00:12<00:00,  8.41it/s]


(6731, 384) 6731


In [7]:
# --- SEE ONE EMBEDDING ---
print(records[0]["embed_text"])
print(embeddings[0][:8])      # first 8 of the 384 numbers
print(embeddings[0].shape)    # (384,)

Roasted Peppers And Mushroom Tortilla Pizza Recipe. Ingredients: tortillas, extra virgin olive oil, garlic, mozzarella cheese, red yellow or green bell pepper, onions, kalmatta olives, button mushrooms
[-0.06160494  0.01275798 -0.02440856  0.03865961 -0.04999339  0.04431468
 -0.05003772 -0.06328132]
(384,)


In [8]:
# --- TEST RETRIEVAL ---
def search(query, k=5):
    q = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    for score, i in zip(scores[0], ids[0]):
        r = records[i]
        print(f"{score:.3f} | {r['title']}")
        print("    ", ", ".join(r["ingredients_clean"]))
        print()

search("tomato, garlic, tamarind, curry leaves")

0.668 | Hyderabadi Bagare Tamatar (Spiced, Tossed Tomato Curry)
     tomatoes, onions, khoya, sesame seeds, raw peanuts, red chilli powder, kashmiri red chilli powder, turmeric powder, coriander powder, cumin powder, cloves, cinnamon stick, garlic, oil, salt, whole eggs

0.666 | Sweet Spicy & Tangy Vegetarian Thai Green Curry Recipe With Tamarind
     lemongrass, garlic, ginger, green chillies, coriander leaves, basil leaves, spring onion, coriander seeds, whole black peppercorns, salt, broccoli, small brinjal, button mushrooms, red bell pepper, green zucchini, brown sugar, basil leaves, salt, sunflower oil

0.650 | Cauliflower and Vegetable Curry (With Almonds and Spices)
     cauliflower, carrots, green beans, potatoes, ginger, onion, garlic, badam, turmeric powder, red chilli powder, fresh cream, salt, coriander leaves

0.621 | Tamatar Chaat Banarasi Style Recipe
     potatoes, red chilli powder, tomatoes, black salt, cumin powder, black pepper powder, tomato ketchup, sugar, onion, 

In [9]:
# --- HYBRID SEARCH ---
def parse_query(query):
    return [x.strip().lower() for x in query.split(",") if x.strip()]

def hybrid_search(query, k=5, pool=50, alpha=0.3, beta=0.02, verbose=True):
    user_ings = parse_query(query)
    q = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, pool)

    results = []
    for emb_score, i in zip(scores[0], ids[0]):
        r = records[i]
        recipe_text = " ".join(r["ingredients_clean"])
        matched = [u for u in user_ings if u in recipe_text]
        coverage = len(matched) / len(user_ings)      # share of YOUR ingredients used
        missing = len(r["ingredients_clean"]) - len(matched)
        final = alpha * emb_score + (1 - alpha) * coverage - beta * missing
        results.append((final, emb_score, coverage, missing, r, matched))

    results.sort(key=lambda x: x[0], reverse=True)
    if not verbose:
        return results[:k]
    for final, emb, cov, missing, r, matched in results[:k]:
        print(f"{final:.3f} | {r['title']}")
        print(f"    you have: {matched} | other ingredients needed: {missing}")
        print()

hybrid_search("tomato, garlic, tamarind, curry leaves")

0.714 | Thandu Keerai Puli Kuzhambu Recipe - Amaranth Greens in Tangy Curry
    you have: ['tomato', 'garlic', 'tamarind', 'curry leaves'] | other ingredients needed: 8

0.694 | Cauliflower Leaves Chutney Recipe
    you have: ['tomato', 'garlic', 'tamarind', 'curry leaves'] | other ingredients needed: 9

0.656 | Andhra Style Zucchini Tomato Chutney Recipe
    you have: ['tomato', 'garlic', 'tamarind', 'curry leaves'] | other ingredients needed: 11

0.539 | Tamatar Wali Arbi Sabzi Recipe
    you have: ['tomato', 'garlic', 'tamarind'] | other ingredients needed: 8

0.503 | Tamatar Pyaz Ki Sabzi Recipe -Tomato Onion Sabzi
    you have: ['tomato', 'garlic', 'curry leaves'] | other ingredients needed: 10



In [10]:
hybrid_search("tomato, garlic, tamarind, curry leaves")

0.714 | Thandu Keerai Puli Kuzhambu Recipe - Amaranth Greens in Tangy Curry
    you have: ['tomato', 'garlic', 'tamarind', 'curry leaves'] | other ingredients needed: 8

0.694 | Cauliflower Leaves Chutney Recipe
    you have: ['tomato', 'garlic', 'tamarind', 'curry leaves'] | other ingredients needed: 9

0.656 | Andhra Style Zucchini Tomato Chutney Recipe
    you have: ['tomato', 'garlic', 'tamarind', 'curry leaves'] | other ingredients needed: 11

0.539 | Tamatar Wali Arbi Sabzi Recipe
    you have: ['tomato', 'garlic', 'tamarind'] | other ingredients needed: 8

0.503 | Tamatar Pyaz Ki Sabzi Recipe -Tomato Onion Sabzi
    you have: ['tomato', 'garlic', 'curry leaves'] | other ingredients needed: 10



In [11]:
# --- TEST OLLAMA FROM PYTHON ---
import ollama

response = ollama.chat(
    model="llama3.2",
    messages=[{"role": "user", "content": "Suggest one simple dish using tomatoes and garlic, in two sentences."}],
)
print(response["message"]["content"])

A simple and delicious dish that combines tomatoes and garlic is Bruschetta, which involves toasting slices of bread, topping them with a mixture of diced tomatoes, minced garlic, olive oil, and basil, and serving immediately. This classic Italian appetizer is easy to make and requires minimal ingredients, making it a great option for a quick and flavorful meal.


In [12]:
# --- ANSWER WITH LLM ---
import ollama

def build_context(results):
    blocks = []
    for n, (final, emb, cov, missing, r, matched) in enumerate(results, 1):
        need = list(dict.fromkeys(c for c in r["ingredients_clean"] if not any(m in c for m in matched)))
        blocks.append(
            f"Recipe {n}: {r['title']}\n"
            f"Ingredients the user has: {', '.join(matched)}\n"
            f"Other ingredients needed ({len(need)}): {', '.join(need)}\n"
            f"Steps: {' '.join(r['steps'])}"
        )
    return "\n\n".join(blocks)

def summarize(r):
    steps = [s for s in r["steps"] if len(s.split()) >= 4]   # drops stray fragments like "or" and "."
    prompt = f"""Summarize the steps below in at most 3 short sentences, in your own words.
Start directly with the first action. Do not write an introduction.
Only describe actions that appear in the steps. Do not add serving suggestions, garnishes, meal times or extra steps. If the steps end early, end your summary at the same point.

Steps: {' '.join(steps)}"""
    response = ollama.chat(
        model="llama3.2",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},
    )
    text = response["message"]["content"].strip()
    lines = [l for l in text.split("\n") if l.strip()]
    if lines and lines[0].lower().startswith("here"):
        lines = lines[1:]
    return " ".join(lines)

def answer(query, k=3):
    results = hybrid_search(query, k=k, verbose=False)
    blocks = []
    for final, emb, cov, missing, r, matched in results:
        need = list(dict.fromkeys(
            c for c in r["ingredients_clean"] if not any(m in c for m in matched)
        ))
        blocks.append(
            f"**{r['title']}**\n"
            f"- You have: {', '.join(matched)}\n"
            f"- You still need ({len(need)} items): {', '.join(need)}\n"
            f"- How to make it: {summarize(r)}"
        )
    return "\n\n".join(blocks)

print(answer("tomato, garlic, tamarind, curry leaves"))

**Thandu Keerai Puli Kuzhambu Recipe - Amaranth Greens in Tangy Curry**
- You have: tomato, garlic, tamarind, curry leaves
- You still need (7 items): green amaranth leaves, onion, mustard seeds, asafoetida, methi powder, sesame oil, salt
- How to make it: Soak tamarind in hot water, then squeeze the pulp out and set it aside. Heat sesame oil in a pan, add mustard seeds, asafoetida, and curry leaves, then sauté until they crackle and splutter. Add garlic, onions, tomatoes, and thandu keerai, then simmer the mixture with tamarind pulp, spices, and salt until it thickens.

**Cauliflower Leaves Chutney Recipe**
- You have: tomato, garlic, tamarind, curry leaves
- You still need (9 items): cauliflower leaves, white urad dal, dry red chilli, salt, asafoetida, turmeric powder, sunflower oil, mustard seeds, cumin seeds
- How to make it: Select the innermost layer of cauliflower leaves and wash them thoroughly. Fry urad dal and red chilli in oil until the urad dal turns golden, then cook the c

In [13]:
# --- CLEAN STRAY STEP FRAGMENTS ---
for r in records:
    r["steps"] = [s for s in r["steps"] if len(s.split()) >= 4]

# re-run the check
r = next(x for x in records if x["title"].startswith("Andhra Style Zucchini Tomato Chutney"))
for i, s in enumerate(r["steps"], 1):
    print(i, s)

1 To begin making the Andhra Style Zucchini Tomato Chutney Recipe first warm some oil in a pan on medium heat.
2 Add the green chilies to it and toss lightly for a minute till they are scalded, and transfer to a plate.
3 Now add garlic pods and zucchini pieces into the pan and sauté till soft and light brown and transfer to the plate too. This may take a couple of minutes.
4 Add the sliced tomatoes along with tamarind and toss together for a minute. Turn off the heat and add minced coriander and stir till wilted.
5 Once the mixture cools down, grind all the above ingredients together along with salt and cumin seeds in a blender to a coarse paste and transfer to a bowl.
6 For the tempering, heat oil in a small pan on medium heat and add mustard, cumin and red chilies. Once they splutter add asafoetida and finally add curry leaves and transfer it to the zucchini chutney.
7 Stir well and serve the Andhra Style Zucchini Tomato Chutney immediately with


In [14]:
# --- CHECK ANDHRA STEPS ---
r = next(x for x in records if x["title"].startswith("Andhra Style Zucchini Tomato Chutney"))
for i, s in enumerate(r["steps"], 1):
    print(i, s)

1 To begin making the Andhra Style Zucchini Tomato Chutney Recipe first warm some oil in a pan on medium heat.
2 Add the green chilies to it and toss lightly for a minute till they are scalded, and transfer to a plate.
3 Now add garlic pods and zucchini pieces into the pan and sauté till soft and light brown and transfer to the plate too. This may take a couple of minutes.
4 Add the sliced tomatoes along with tamarind and toss together for a minute. Turn off the heat and add minced coriander and stir till wilted.
5 Once the mixture cools down, grind all the above ingredients together along with salt and cumin seeds in a blender to a coarse paste and transfer to a bowl.
6 For the tempering, heat oil in a small pan on medium heat and add mustard, cumin and red chilies. Once they splutter add asafoetida and finally add curry leaves and transfer it to the zucchini chutney.
7 Stir well and serve the Andhra Style Zucchini Tomato Chutney immediately with


In [15]:
# --- GRADIO CHAT ---
import gradio as gr

def chat(message, history):
    if "," not in message:
        return "Please list your ingredients separated by commas, e.g. tomato, garlic, tamarind."
    return answer(message, k=3)

demo = gr.ChatInterface(
    fn=chat,
    title="Recipe Finder",
    description="Type the ingredients you have, separated by commas.",
)
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
